# CASL-VAE: a cross-sectional 20-ROI example

Run the cells in order in the `caslvae` environment. This notebook uses the **original model and training loop in `code/`**. No covariate correction, covariate conditioning, temporal consistency, GRU, or flow is added.

We reuse the longitudinal simulator but select **one scan per subject**. Ground-truth subtype and severity are available for evaluation only. The original categorical salient encoder remains part of CASL-VAE; no additional subtype-prediction model is introduced.

Training runs for up to 500 epochs using the original stopping rule. Results depend on initialization and the simulated disease burden; this notebook demonstrates the workflow rather than guarantees subtype recovery.


In [ ]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader

# Works when opened from the repository root or docs/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "code" / "CASLVAE_model.py").is_file())
sys.path.insert(0, str(ROOT / "code"))
sys.path.insert(0, str(ROOT / "longitudinal"))
from caslvae_synthetic_data import generate_temporal_roi_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT = ROOT / "outputs" / "caslvae_cross_sectional_demo"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)


## 1. Generate cross-sectional samples

The simulator creates individual anatomy, normal aging, two non-overlapping disease patterns, and measurement noise. Each disease pattern affects 6 of the 20 ROIs. Disease effect equals the pattern multiplied by simulated severity.

One randomly selected scan represents each subject. A converter sampled before onset is labeled `Background`, even if disease appears at a later visit. Age/sex/ICV variation remains in the observations because we disable covariate correction.


In [ ]:
subjects, population = generate_temporal_roi_dataset(n_subjects=600, n_rois=20, seed=SEED)
rng = np.random.default_rng(SEED + 1)
features = [f"ROI_{j+1:02d}" for j in range(20)]
records, true_effects, true_healthy = [], [], []
for subject in subjects:
    visit = int(rng.integers(len(subject["x"])))
    truth = subject["truth"]
    active = bool(truth["visit_target"][visit])
    records.append(dict(
        subject_id=subject["id"], label="Target" if active else "Background",
        true_category=int(truth["subtype"]) if active else 0,
        age=float(subject["age"][visit]), severity=float(truth["severity"][visit]),
        **dict(zip(features, subject["x"][visit])),
    ))
    true_effects.append(truth["deviation"][visit])
    true_healthy.append(truth["healthy"][visit])

data = pd.DataFrame(records)
true_effects = np.asarray(true_effects)
true_healthy = np.asarray(true_healthy)

# Subject-disjoint partitions: one row already corresponds to one subject.
train_index, heldout_index = train_test_split(
    np.arange(len(data)), test_size=0.30, random_state=SEED,
    stratify=data["true_category"],
)
val_index, test_index = train_test_split(
    heldout_index, test_size=0.50, random_state=SEED,
    stratify=data.iloc[heldout_index]["true_category"],
)
data["split"] = ""
for name, indices in [("train", train_index), ("validation", val_index), ("test", test_index)]:
    data.loc[indices, "split"] = name
data.to_csv(OUTPUT / "synthetic_cross_sectional.csv", index=False)
np.savez_compressed(OUTPUT / "synthetic_truth.npz",
                    disease_effect=true_effects, healthy=true_healthy)
display(pd.crosstab(data["split"], data["true_category"]).rename(
    columns={0: "Reference", 1: "Target subtype 1", 2: "Target subtype 2"}))


In [ ]:
colors = ["tab:blue", "tab:orange", "tab:green"]
names = ["Reference", "Target subtype 1", "Target subtype 2"]
patterns = np.stack([population["disease_pattern1"], population["disease_pattern2"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for k, ax in enumerate(axes):
    ax.bar(np.arange(1, 21), patterns[k], color=colors[k+1])
    ax.set(title=f"True target pattern {k+1}: severity = 1",
           xlabel="ROI", ylabel="Disease effect (volume units)", xticks=np.arange(1, 21, 2))
fig.tight_layout()
fig.savefig(OUTPUT / "true_patterns.png", dpi=150)
plt.show()

# Include one affected ROI per subtype and two unaffected ROIs.
affected = list(population["pattern1_rois"]) + list(population["pattern2_rois"])
unaffected = [j for j in range(20) if j not in affected]
example_rois = [int(population["pattern1_rois"][0]),
                int(population["pattern2_rois"][0]), *unaffected[:2]]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, roi in zip(axes, example_rois):
    for k in range(3):
        values = data.loc[data["true_category"].eq(k), features[roi]]
        ax.hist(values, bins=25, density=True, alpha=.45, color=colors[k], label=names[k])
    ax.set(title=features[roi], xlabel="Observed volume", ylabel="Density")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "population_roi_distributions.png", dpi=150)
plt.show()


## 2. Create the original CASL-VAE

As in the research scripts: import `CASLVAE`, preprocess the data, build unpaired datasets, call `create(opt)`, and attach preprocessing parameters.

Only the 20 ROI measurements enter the model. Preprocessing is fitted on **training reference samples only**. Subtype labels are used to balance simulation partitions and visualize results, never to train the model.


In [ ]:
from CASLVAE_model import CASLVAE
from data_utils import PreprocessingStats, PreprocessingTransform, CreateDataset
from train_utils import train_model

train_df = data.iloc[train_index]
stats = PreprocessingStats(
    train_df[features].to_numpy(), train_df["label"].to_numpy(),
    covariate_correct=False,
)
stats.stats()

def transform(indices):
    return PreprocessingTransform(
        data.iloc[indices][features].to_numpy(), stats, covariate_correct=False,
    ).transform().astype(np.float32)

train_x, val_x, test_x = [transform(i) for i in (train_index, val_index, test_index)]
train_dataset = CreateDataset(train_x, data.iloc[train_index]["label"].to_numpy())
val_dataset = CreateDataset(val_x, data.iloc[val_index]["label"].to_numpy())

opt = dict(
    nfeatures=20, nLatent_z=15, nLatent_s_cont=2,
    nLatent_s_cat=1, ncategories=3,
    mmd_lambda=100, kl_z_lambda=1, kl_scat_lambda=1,
    kl_scont_lambda=1, kl_sconty_lambda=1,
    consistency_lambda=10, entropy_lambda=1, norm_salienty_lambda=0,
    lr=.001, beta1=.5, scheduler=False, decay_epochs=1,
    DEVICE=DEVICE, epochs=500, batch_size=128, path=str(OUTPUT),
)
loader = DataLoader(train_dataset, batch_size=opt["batch_size"], shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
model = CASLVAE()
model.create(opt=opt)
model.set_preprocessing_params(dict(
    data_mean=stats.data_mean, data_std=stats.data_std,
    cov_min=None, cov_max=None, cov_beta=None, covariate_correct=False,
    feature_names=features, covariate_names=[], tau=1.,
))
print(type(model.G_forward).__name__)
print("Features only; no covariates or subtype ground truth supplied.")


## 3. Train

This cell performs training; skip it if a checkpoint already exists. The original loop saves `model_checkpoint.pt` when its stopping rule triggers or the epoch budget ends. It does not restore the minimum-BCE or minimum-reconstruction epoch. The next cell reloads what was actually saved.

The loop's diagnostic panels assume at least eight input features; this example has twenty.


In [ ]:
def tensor(values):
    return torch.as_tensor(values, dtype=torch.float32, device=DEVICE)

model, clustering_logits, loss_history, converged = train_model(
    model, opt, loader,
    tensor(train_dataset.data_bg), tensor(train_dataset.data_tgt),
    tensor(val_dataset.data_bg), tensor(val_dataset.data_tgt),
)
print("Original stopping rule triggered:", converged)


## 4. Load the checkpoint and evaluate held-out subjects

Inference uses common posterior means, hard categorical assignments, and the continuous salient encoder output. These are deterministic summaries, unlike sampled latents used during training. Reference samples use the model's reference salient preset for reconstruction; target samples use inferred codes.

The same common latent with the reference salient preset gives a paired reference estimate. It is not guaranteed to equal the simulator's true individual healthy anatomy.


In [ ]:
model = CASLVAE()
checkpoint = model.load_model(str(OUTPUT / "model_checkpoint.pt"), device=DEVICE)
g = model.G_forward
g.eval()

# Reapply saved preprocessing rather than fitting anything on test subjects.
def saved_transform(indices):
    raw = data.iloc[indices][model.feature_names].to_numpy()
    return ((raw - model.train_mean) / (model.train_std + 1e-6)).astype(np.float32)

with torch.no_grad():
    x = tensor(saved_transform(test_index))
    z, _ = g.encode_common(x)
    logits, salient_features = g.encode_salient_cat(x)
    probabilities = logits.softmax(-1)
    inferred_category = probabilities.argmax(-1)
    code = torch.nn.functional.one_hot(inferred_category, opt["ncategories"]).to(x.dtype)
    reference_code = torch.zeros_like(code)
    reference_code[:, 0] = 1
    is_reference = tensor(data.iloc[test_index]["label"].eq("Background").to_numpy()).bool()
    reconstruction_code = torch.where(is_reference[:, None], reference_code, code)
    s_cont = g.enc_s_cont(torch.cat([salient_features, reconstruction_code], dim=1))
    reconstruction_s = torch.where(is_reference[:, None], torch.zeros_like(s_cont), s_cont)
    common = g.decode_common(z)
    reconstruction = common + g.decode_salient(reconstruction_code, reconstruction_s)
    paired_reference = common + g.decode_salient(reference_code, torch.zeros_like(s_cont))

scale = np.asarray(model.train_std) + 1e-6
reconstruction_raw = reconstruction.cpu().numpy() * scale + model.train_mean
paired_raw = paired_reference.cpu().numpy() * scale + model.train_mean
observed = data.iloc[test_index][features].to_numpy()
truth = data.iloc[test_index]["true_category"].to_numpy()
print("Saved epoch (loop index):", checkpoint["epoch"], "| temperature:", checkpoint["tau"])
print("Test reconstruction MAE:", round(float(abs(reconstruction_raw-observed).mean()), 2))
print("Paired reference vs true healthy MAE:",
      round(float(abs(paired_raw-true_healthy[test_index]).mean()), 2))
np.savez_compressed(OUTPUT / "test_outputs.npz",
                    subject_id=data.iloc[test_index]["subject_id"].to_numpy(dtype=str),
                    probabilities=probabilities.cpu().numpy(), z=z.cpu().numpy(),
                    s_cont=s_cont.cpu().numpy(), reconstruction=reconstruction_raw,
                    paired_reference=paired_raw, true_category=truth)


## 5. Prototypical target patterns

For each target category, decode its learned continuous salient **prior mean** and subtract the reference salient preset. The common term cancels. Multiply differences by the saved feature standard deviations; do not add the feature mean.

The ground-truth patterns above are defined at severity 1. Learned prior prototypes have their own learned magnitude, so their amplitudes need not match. Category 1/2 labels can be permuted relative to the simulated subtypes.


In [ ]:
with torch.no_grad():
    target_codes = torch.eye(opt["ncategories"], device=DEVICE)[1:]
    prior_s = g.prior_s_cont(target_codes)
    ref_codes = torch.zeros_like(target_codes)
    ref_codes[:, 0] = 1
    prototypes = (g.decode_salient(target_codes, prior_s)
                  - g.decode_salient(ref_codes, torch.zeros_like(prior_s)))
prototypes = prototypes.cpu().numpy() * scale

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for k, ax in enumerate(axes):
    ax.bar(np.arange(1, 21), prototypes[k], color=colors[k+1])
    ax.axhline(0, color="black", lw=.7)
    ax.set(title=f"Learned target category {k+1}: prior prototype",
           xlabel="ROI", ylabel="Deviation from reference (volume units)",
           xticks=np.arange(1, 21, 2))
fig.tight_layout()
fig.savefig(OUTPUT / "learned_prototypes.png", dpi=150)
plt.show()


## 6. Inspect continuous salient and common representations

The common latent is 15-dimensional; PCA is fitted on **training common embeddings** and applied to test embeddings. The salient latent is already two-dimensional. Reference salient codes are preset to zero by the model; the salient scatter therefore shows target subjects only.

Coloring by simulator truth helps assess separation and leakage; it does not supervise training. Neither a PCA plot nor visual separation proves disentanglement.


In [ ]:
with torch.no_grad():
    train_z, _ = g.encode_common(tensor(saved_transform(train_index)))
pca = PCA(n_components=2).fit(train_z.cpu().numpy())
z_plot = pca.transform(z.cpu().numpy())
s_plot = s_cont.cpu().numpy()
target_mask = truth > 0
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k in range(3):
    mask = truth == k
    axes[0].scatter(z_plot[mask, 0], z_plot[mask, 1], c=colors[k],
                    marker=["o", "^", "s"][k], alpha=.65, label=names[k])
    if k:
        mask = mask & target_mask
        axes[1].scatter(s_plot[mask, 0], s_plot[mask, 1], c=colors[k],
                        marker=["o", "^", "s"][k], alpha=.65, label=names[k])
axes[0].set(title="Test common z (training PCA)", xlabel="PC1", ylabel="PC2")
axes[1].set(title="Test target continuous salient code", xlabel="s_cont 1", ylabel="s_cont 2")
for ax in axes[:2]:
    ax.legend(fontsize=8)
scatter = axes[2].scatter(z_plot[:, 0], z_plot[:, 1],
                          c=data.iloc[test_index]["age"], cmap="viridis", alpha=.7)
axes[2].set(title="Common z colored by age (not an input)", xlabel="PC1", ylabel="PC2")
fig.colorbar(scatter, ax=axes[2], label="Simulated age")
fig.tight_layout()
fig.savefig(OUTPUT / "test_latent_representations.png", dpi=150)
plt.show()


All outputs are under `outputs/caslvae_cross_sectional_demo/`. Ground-truth effects, subtype labels, and healthy counterparts are retained for evaluation only. This notebook does not perform longitudinal forecasting or change the original CASL-VAE objective.

See [the experiment guide](experiments.md) for category-count selection and training behavior, and [the usage guide](usage.md) for generation and output interpretation.
